In [1]:
!pip install -q transformers peft trl bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 13.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# 1. Define the AyurParam model ID
model_id = "bharatgenai/AyurParam"

# Load the configuration explicitly and disable rope_scaling to avoid the KeyError
config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
config.rope_scaling = None 

# 2. Configure QLoRA (4-bit quantization for memory efficiency)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 3. Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

config_parambharatgen.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/bharatgenai/AyurParam:
- config_parambharatgen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.8M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [4]:
import os

for root, dirs, files in os.walk('/kaggle'):
    for file in files:
        if "charaka" in file:
            print(os.path.join(root, file))

/kaggle/input/datasets/gowthamkommana/dataset1/train_ready_charaka.jsonl


In [5]:
# Add padding token if it doesn't exist
if getattr(tokenizer, "pad_token", None) is None:
    tokenizer.pad_token = tokenizer.eos_token

# Pass the patched config into the model loader
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    quantization_config=bnb_config,
    device_map={"": 0},   # ✅ force single GPU
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

# 4. Set up LoRA configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)
model = get_peft_model(model, peft_config)

model.to("cuda:0")

# 5. Load your flattened JSONL dataset
dataset = load_dataset("json", data_files="/kaggle/input/datasets/gowthamkommana/dataset1/train_ready_charaka.jsonl", split="train")

training_args = SFTConfig(
    output_dir="./ayurparam_charaka_finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    num_train_epochs=3,
    logging_steps=10,
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    save_strategy="epoch"
)

modeling_parambharatgen.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/bharatgenai/AyurParam:
- modeling_parambharatgen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args
)

Adding EOS to train dataset:   0%|          | 0/292 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/292 [00:00<?, ? examples/s]

In [7]:
print("Starting fine-tuning...")
trainer.train() 

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 256005, 'pad_token_id': 3}.


Starting fine-tuning...


You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,11.801588
20,9.479712
30,8.651772
40,8.357225
50,8.256641
60,8.163393
70,7.969001
80,7.875410
90,7.852162
100,7.764847


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=111, training_loss=8.528853717150989, metrics={'train_runtime': 4467.6397, 'train_samples_per_second': 0.196, 'train_steps_per_second': 0.025, 'total_flos': 8679040250585088.0, 'train_loss': 8.528853717150989})

In [8]:
trainer.model.save_pretrained("./ayurparam_charaka_adapter")  
tokenizer.save_pretrained("./ayurparam_charaka_adapter")      
print("Model fine-tuning complete and saved!")                

Model fine-tuning complete and saved!


In [9]:
import shutil

# Zip fine-tuned model
shutil.make_archive(
    "ayurparam_charaka_finetuned",  # zip name
    'zip',
    "./ayurparam_charaka_finetuned"
)

# Zip adapter model
shutil.make_archive(
    "ayurparam_charaka_adapter",
    'zip',
    "./ayurparam_charaka_adapter"
)

print("✅ Zipping complete!")

✅ Zipping complete!


In [10]:
import os

for root, dirs, files in os.walk('/kaggle'):
    for file in files:
        if "charaka" in file:
            print(os.path.join(root, file)) 

/kaggle/input/datasets/gowthamkommana/dataset1/train_ready_charaka.jsonl
/kaggle/working/ayurparam_charaka_finetuned.zip
/kaggle/working/ayurparam_charaka_adapter.zip
